In [1]:
!pip install opencv-python transformers torch torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 79.6 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

In [2]:
from google.colab import files

# Upload video manually
uploaded = files.upload()


Saving video.mp4 to video.mp4


In [4]:
import cv2
import os

os.makedirs("frames", exist_ok=True)

cap = cv2.VideoCapture("video.mp4")
fps = cap.get(cv2.CAP_PROP_FPS)
frame_interval = int(fps * 2)  # Capture a frame every 2 seconds

frame_count = 0
saved_count = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    if frame_count % frame_interval == 0:
        frame_filename = f"frames/frame_{saved_count}.jpg"
        cv2.imwrite(frame_filename, frame)
        saved_count += 1
    frame_count += 1

cap.release()
print(f"✅ Extracted {saved_count} frames.")


✅ Extracted 30 frames.


In [5]:
from transformers import VisionEncoderDecoderModel, ViTImageProcessor, AutoTokenizer
from PIL import Image
import torch
import glob

# Load pretrained model
model = VisionEncoderDecoderModel.from_pretrained("nlpconnect/vit-gpt2-image-captioning")
feature_extractor = ViTImageProcessor.from_pretrained("nlpconnect/vit-gpt2-image-captioning")
tokenizer = AutoTokenizer.from_pretrained("nlpconnect/vit-gpt2-image-captioning")

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# Frame captioning function
def generate_caption(image_path):
    image = Image.open(image_path).convert("RGB")
    pixel_values = feature_extractor(images=[image], return_tensors="pt").pixel_values.to(device)
    output_ids = model.generate(pixel_values, max_length=16, num_beams=4)
    caption = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return caption

# Generate captions for all frames
frame_paths = sorted(glob.glob("frames/*.jpg"))
captions = []

for path in frame_paths:
    caption = generate_caption(path)
    captions.append(caption)

print("✅ Captions generated for all frames.")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/4.61k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


pytorch_model.bin:   0%|          | 0.00/982M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


...

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/120 [00:00<?, ?B/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.
You may ignore this warning if your `pad_token_id` (50256) is identical to the `bos_token_id` (50256), `eos_token_id` (50256), or the `sep_token_id` (None), and your input is not padded.


✅ Captions generated for all frames.


In [6]:
from transformers import pipeline

# Load summarization pipeline
summarizer = pipeline("summarization")

# Combine all captions into a single paragraph
video_description = " ".join(captions)
print("📝 Frame Captions:\n", video_description)

# Generate summary
summary = summarizer(video_description, max_length=60, min_length=20, do_sample=False)[0]['summary_text']

print("\n✅ Video Summary:\n", summary)


No model was supplied, defaulted to sshleifer/distilbart-cnn-12-6 and revision a4f8f3e (https://huggingface.co/sshleifer/distilbart-cnn-12-6).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/1.80k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

Device set to use cpu


📝 Frame Captions:
 a highway filled with lots of cars and trucks  a street filled with lots of cars and trucks  a city street filled with lots of traffic  a city street filled with lots of traffic  a city street filled with lots of traffic  a street filled with lots of cars and trucks  a street filled with lots of cars and trucks  a street filled with lots of cars and trucks  a street filled with lots of cars and trucks  a street filled with lots of cars and trucks  a street filled with lots of cars and trucks  a city street filled with lots of traffic  a street filled with lots of cars and trucks  a street filled with lots of cars and trucks  a city street filled with lots of traffic  a city street filled with lots of cars and trucks  a city street filled with lots of traffic  a street filled with lots of cars and trucks  a highway filled with lots of traffic  a street filled with lots of cars and trucks  a highway filled with lots of cars and trucks  a street filled with lots of cars

In [12]:
# Step 1: Install
!pip install rouge-score nltk

# Step 2: Import
from rouge_score import rouge_scorer
import nltk
nltk.download('punkt')

# Step 3: Reference and Hypothesis
reference_summary = "A city highway filled with cars and trucks"
generated_summary = summary  # from your model

# Step 4: Compute ROUGE
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
scores = scorer.score(reference_summary, generated_summary)

print("Evaluation Metrics:")
for key in scores:
    print(f"{key}: {scores[key]}")


Evaluation Metrics:
rouge1: Score(precision=0.17391304347826086, recall=1.0, fmeasure=0.29629629629629634)
rouge2: Score(precision=0.15555555555555556, recall=1.0, fmeasure=0.2692307692307693)
rougeL: Score(precision=0.15217391304347827, recall=0.875, fmeasure=0.2592592592592593)


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
